# YouTube video scraping

This notebook is used to get YouTube comments from: https://www.youtube.com/watch?v=V79x7045Bp0

## Import Important Libraries

In [1]:
import time

from selenium.webdriver import Chrome
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
import pandas as pd

## Define Empty Data List

In [2]:
data = []

## Define Target Url and Services

In [3]:
url= "https://www.youtube.com/watch?v=V79x7045Bp0"
service = Service(executable_path=r'C:\Users\steam\OneDrive\Desktop\Coding Kuliah\Python\Text Mining\.env\chromedriver\chromedriver.exe')
timeout = 15 # 15 seconds of timeout
sleep_time = 10

## Web Scraping

In [4]:
with Chrome(service=service) as driver:

    wait = WebDriverWait(driver, timeout)
    driver.get(url)
 
    previous_height = driver.execute_script("return document.documentElement.scrollHeight")

    while True:

        wait.until(EC.visibility_of_element_located((By.TAG_NAME, "body"))).send_keys(Keys.END)
        time.sleep(sleep_time)

        new_height = driver.execute_script("return document.documentElement.scrollHeight")

        if (previous_height == new_height):
             break
        
        previous_height = new_height

    # Find the button that contains "repl" by XPath (or any other method)
    for reply_button in wait.until(EC.presence_of_all_elements_located((By.XPATH, '//button[contains(@aria-label, "repl")]'))):
        driver.execute_script('arguments[0].scrollIntoView(true)', reply_button)
        driver.execute_script('arguments[0].click()', reply_button)
        time.sleep(3)

    wait.until(EC.visibility_of_element_located((By.TAG_NAME, "body"))).send_keys(Keys.END)

    for comment in wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#content-text"))):
            data.append(comment.text.replace('\n', '\\n')) # Replace '\n' with '\\n'
            print(comment.text)

## Save the data as dataframe

In [5]:
df = pd.DataFrame(data, columns=['comment'])

## Remove missing values immediately

In [6]:
df = df.replace('', pd.NA)
df = df.dropna()

## Save the data to tsv

Since this is a text data, there are probably many usages of ',' in the data

In [7]:
df.to_csv('../Data/RichardWalls.tsv', index=False, sep='\t')